### html + htmlheadertextsplitter

In [1]:
import sys
import os
import chromadb
path = "C:\\Users\\luyer\\my_rag\\data"

In [2]:
client = chromadb.PersistentClient(path=path)

In [3]:
collection_persistent = client.get_or_create_collection(name="my_movies_html")

In [4]:
client_regular = chromadb.Client()

In [5]:
collection_regular = client_regular.create_collection(name="my_movies_html_regular")

In [6]:
files = [
    file_name
    for file_name in os.listdir(path)
    if os.path.isfile(os.path.join(path, file_name)) and file_name.endswith('.html')
]
print(files)

['2001_A_Space_Odyssey.html', 'Casablanca.html', 'Citizen_Kane.html', 'Parasite_2019.html', 'Pulp_Fiction.html', 'Seven_Samurai.html', 'Spirited_Away.html', 'The_Dark_Knight.html', 'The_Godfather.html', 'The_Matrix.html']


In [7]:
import re

def clean_chunk_text(text: str) -> str:
    """Normalize whitespace and remove unwanted control characters."""
    text = text.replace("\x00", "")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [8]:
from bs4 import BeautifulSoup
from pathlib import Path
from langchain_text_splitters import HTMLHeaderTextSplitter

html_splitter = HTMLHeaderTextSplitter(
    headers_to_split_on=[
        ("h1", "header1"),
        ("h2", "header2"),
        ("h3", "header3"),
    ]
)

all_chunks = []

for file_name in files[:1]:
    file_path = Path(path) / file_name

    html = file_path.read_text(encoding="utf-8", errors="ignore")
    chunks = html_splitter.split_text(html)

    for chunk_index, chunk in enumerate(chunks):
        text = clean_chunk_text(chunk.page_content)

        if not text:
            continue

        all_chunks.append(
            {
                "id": f"html-{file_name}-{chunk_index}",
                "document": text,
                "metadata": {
                    **chunk.metadata,
                    "source": file_name,
                },
            }
        )

collection_persistent.add(
    ids=[item["id"] for item in all_chunks],
    documents=[item["document"] for item in all_chunks],
    metadatas=[item["metadata"] for item in all_chunks],
)

print(f"Added {len(all_chunks)} HTML chunks")

Added 81 HTML chunks


In [10]:
import pprint

In [12]:
pprint.pprint(chunk.page_content)

('.mw-parser-output .sister-box .side-box-abovebelow{padding:0.75em '
 '0;text-align:center}.mw-parser-output .sister-box '
 '.side-box-abovebelow>b{display:block}.mw-parser-output .sister-box '
 '.side-box-text>ul{border-top:1px solid #aaa;padding:0.75em '
 '0;width:220px;margin:0 auto}.mw-parser-output .sister-box '
 '.side-box-text>ul>li{min-height:31px}.mw-parser-output '
 '.sister-logo{display:inline-block;width:31px;line-height:31px;vertical-align:middle;text-align:center}.mw-parser-output '
 '.sister-link{display:inline-block;margin-left:7px;width:182px;vertical-align:middle}@media '
 'print{body.ns-0 .mw-parser-output '
 '.sistersitebox{display:none!important}}@media '
 'screen{html.skin-theme-clientpref-night .mw-parser-output .sistersitebox '
 'img[src*="Wiktionary-logo-v2.svg"]{background-color:white}}@media screen and '
 '(prefers-color-scheme:dark){html.skin-theme-clientpref-os .mw-parser-output '
 '.sistersitebox '
 'img[src*="Wiktionary-logo-v2.svg"]{background-color:whi

In [13]:
len(chunks)

81

In [14]:
chunks[0].page_content

'1968 film by Stanley Kubrick  \n.mw-parser-output .hatnote{font-style:italic}.mw-parser-output div.hatnote{padding-left:1.6em;margin-bottom:0.5em}.mw-parser-output .hatnote i{font-style:normal}.mw-parser-output .hatnote+.mw-empty-elt+.hatnote{margin-top:-0.5em}@media print{body.ns-0 .mw-parser-output .hatnote{display:none!important}}  \nThis article is about the 1968 film. For the novel, see . For other uses, see .  \n(novel)  \n2001: A Space Odyssey  \n2001: A Space Odyssey (disambiguation)  \n"A Space Odyssey" redirects here. For other uses, see .  \nSpace Odyssey (disambiguation)  \n@media(min-width:640px){.mw-parser-output .infobox{margin-left:1em;float:right;clear:right;width:22em}}.mw-parser-output .infobox-subbox{padding:0;border:none;margin:-3px;width:auto;min-width:100%;font-size:100%;clear:none;float:none;background-color:transparent;color:inherit}.mw-parser-output .infobox-3cols-child{margin:-3px}.mw-parser-output .infobox .navbar{font-size:100%}.mw-parser-output .infobox-h

In [25]:
type(chunks)

list

In [15]:
results = collection_regular.query(
    query_texts=["What year was 2001: A Space Odyssey released, and what genre is it?"],
)

In [19]:
results['documents'][0][0]

'antasy/2001-a-space-odyssey.html). Audio-Video Revolution. Archived from the original (htt p:// on 14 May 2011. Retrieved 7 January 2011. 138. "2001: A Space Odyssey (Remastered)" ( dvd.net.au. Archived ( cgi?review_id=1126) from the original on 1 February 2011. Retrieved 7 January 2011. 139. Hall, Sheldon (9 April 2011). "Introduction to 2001: A Space Odyssey" ( g/web/20110526092759/ In70mm.com. Archived from the original ( ndex.htm) on 26 May 2011.'

In [16]:
collection_persistent.query(
    query_texts=["What year was 2001: A Space Odyssey released, and what genre is it?"],
)

{'ids': [['html-2001_A_Space_Odyssey.html-37',
   'html-2001_A_Space_Odyssey.html-19',
   'html-2001_A_Space_Odyssey.html-39',
   'html-2001_A_Space_Odyssey.html-50',
   'html-2001_A_Space_Odyssey.html-66',
   'html-2001_A_Space_Odyssey.html-64',
   'html-2001_A_Space_Odyssey.html-17',
   'html-2001_A_Space_Odyssey.html-48',
   'html-2001_A_Space_Odyssey.html-44',
   'html-2001_A_Space_Odyssey.html-58']],
 'embeddings': None,
 'documents': [['Main article: 2001: A Space Odyssey (soundtrack) The initial MGM soundtrack album release contained none of the material from the altered and uncredited rendition of Ligeti\'s used in the film, but used a different recording of (performed by the conducted by ) from that heard in the film, and a longer excerpt of than in the film. In 1996, Turner Entertainment/ released a new soundtrack on CD that included the film\'s rendition of , the version of used in the film, and the shorter version of from the film. As additional "bonus tracks" at the end, t